<a href="https://colab.research.google.com/github/wagueacarinetech-hue/Machine-Learning-project-22D/blob/main/BERT_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install required libraries
!pip install transformers torch scikit-learn pandas numpy


In [2]:
import pandas as pd
import numpy as np
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Load dataset from GitHub
df = pd.read_csv('https://raw.githubusercontent.com/wagueacarinetech-hue/Machine-Learning-project-22D/main/combined_ai_tweet_detection_dataset.csv')

print(f'Dataset loaded! Total rows: {len(df)}')
print(df['label'].value_counts())


Using device: cuda
Dataset loaded! Total rows: 35443
label
1    19796
0    15647
Name: count, dtype: int64


In [3]:
# Split data
X = df['text'].astype(str)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Training set: {len(X_train)} tweets')
print(f'Testing set: {len(X_test)} tweets')

# Load BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
print('Tokenizer loaded!')

Training set: 28354 tweets
Testing set: 7089 tweets


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer loaded!


In [4]:
# Create PyTorch Dataset for BERT
class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Create datasets and dataloaders
train_dataset = TweetDataset(X_train, y_train, tokenizer)
test_dataset = TweetDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f'Training batches: {len(train_loader)}')
print(f'Testing batches: {len(test_loader)}')
print('Dataset ready!')

Training batches: 887
Testing batches: 222
Dataset ready!


In [5]:
# Load BERT model
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)
model = model.to(device)
print('BERT model loaded!')

# Training settings
EPOCHS = 3
optimizer = AdamW(model.parameters(), lr=2e-5)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# Training loop
print('\nStarting training...')
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        scheduler.step()

        if batch_idx % 100 == 0:
            print(f'Epoch {epoch+1}/{EPOCHS} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}')

    avg_loss = total_loss / len(train_loader)
    print(f'\nEpoch {epoch+1} complete! Average loss: {avg_loss:.4f}\n')

print('Training complete!')

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT model loaded!

Starting training...
Epoch 1/3 | Batch 0/887 | Loss: 0.7206
Epoch 1/3 | Batch 100/887 | Loss: 0.4493
Epoch 1/3 | Batch 200/887 | Loss: 0.1581
Epoch 1/3 | Batch 300/887 | Loss: 0.2128
Epoch 1/3 | Batch 400/887 | Loss: 0.1234
Epoch 1/3 | Batch 500/887 | Loss: 0.2218
Epoch 1/3 | Batch 600/887 | Loss: 0.1652
Epoch 1/3 | Batch 700/887 | Loss: 0.1082
Epoch 1/3 | Batch 800/887 | Loss: 0.2451

Epoch 1 complete! Average loss: 0.2365

Epoch 2/3 | Batch 0/887 | Loss: 0.1164
Epoch 2/3 | Batch 100/887 | Loss: 0.1440
Epoch 2/3 | Batch 200/887 | Loss: 0.0972
Epoch 2/3 | Batch 300/887 | Loss: 0.2039
Epoch 2/3 | Batch 400/887 | Loss: 0.0889
Epoch 2/3 | Batch 500/887 | Loss: 0.0102
Epoch 2/3 | Batch 600/887 | Loss: 0.1046
Epoch 2/3 | Batch 700/887 | Loss: 0.1047
Epoch 2/3 | Batch 800/887 | Loss: 0.2197

Epoch 2 complete! Average loss: 0.1269

Epoch 3/3 | Batch 0/887 | Loss: 0.0365
Epoch 3/3 | Batch 100/887 | Loss: 0.0637
Epoch 3/3 | Batch 200/887 | Loss: 0.0156
Epoch 3/3 | Batch 300/

In [6]:
# Evaluating BERT
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print('=== BERT RESULTS ===')
print(f'Accuracy:  {accuracy_score(all_labels, all_preds):.4f}')
print(f'Precision: {precision_score(all_labels, all_preds):.4f}')
print(f'Recall:    {recall_score(all_labels, all_preds):.4f}')
print(f'F1-Score:  {f1_score(all_labels, all_preds):.4f}')
print('\nDetailed Report:')
print(classification_report(all_labels, all_preds, target_names=["Human", "AI Generated"]))

=== BERT RESULTS ===
Accuracy:  0.9209
Precision: 0.9220
Recall:    0.9376
F1-Score:  0.9297

Detailed Report:
              precision    recall  f1-score   support

       Human       0.92      0.90      0.91      3130
AI Generated       0.92      0.94      0.93      3959

    accuracy                           0.92      7089
   macro avg       0.92      0.92      0.92      7089
weighted avg       0.92      0.92      0.92      7089

